# AI Bootcamp — Section 4 · Practical 3: **Graph Pattern**
### Ollama + Qwen3 1.7B · Windows / JupyterLab · No AWS

आपने **Practical 1: Supervisor–Worker** और **Practical 2: Agent as Tool** पूरे किए हैं। अब एक **Graph Workflow** बनाएँगे जो ई-कॉमर्स के ऑर्डर को निश्चित रास्तों से गुज़ारेगा।

**सीखने का लक्ष्य:** Graph = nodes (steps) + directed edges (allowed next steps) + routing decisions. Graph pattern में रास्ता नियमों के अनुसार बदलता है।

**सुरक्षा:** SwiftCart के नीचे दिए गए सारे Orders **काल्पनिक** हैं। कोई असली Payment, Shipping या Refund नहीं होता। मॉडल सिर्फ व्याख्या करेगा; निर्णय सत्यापित Python Rules से होंगे।

**चलाने का तरीका:** सभी **8 Code Cells** क्रम से `Shift + Enter` से चलाएँ। Ollama Windows App शुरू रहनी चाहिए। कोई अतिरिक्त `pip install` या AWS Key नहीं चाहिए।

## Cell 1 — Ollama और Qwen3 Check
अभी बस Ollama Local Server तथा Model के नाम की जाँच करेंगे।

In [ ]:
# Cell 1 — Check local Ollama
import json
from pathlib import Path
from urllib.request import Request, urlopen
from urllib.error import URLError, HTTPError

OLLAMA_URL = "http://127.0.0.1:11434"
MODEL = "qwen3:1.7b"

try:
    with urlopen(f"{OLLAMA_URL}/api/tags", timeout=15) as response:
        model_list = [m.get("name", "") for m in json.load(response).get("models", [])]
except (URLError, TimeoutError, OSError) as exc:
    raise RuntimeError("Ollama से कनेक्शन नहीं हुआ। Windows में Ollama App खोलें।") from exc

print("Installed models:", model_list)
if MODEL not in model_list:
    raise RuntimeError(f"{MODEL} उपलब्ध नहीं है। PowerShell में चलाएँ: ollama pull {MODEL}")
print("PASS: Local Ollama + Qwen3 connected; AWS not required")

## Cell 2 — काल्पनिक Orders और Business Rules
इन 5 Cases से Graph की अलग-अलग Branch Test होगी। **कोई असली Order नहीं** है।

In [ ]:
# Cell 2 — Fictional sample orders and deterministic rules
RULES = {"risk_hold_threshold": 70}
ORDERS = [
    {"order_id": "ORD-101", "payment_confirmed": True,  "risk_score": 12, "fraud_flag": False, "stock": 5, "quantity": 1},
    {"order_id": "ORD-102", "payment_confirmed": True,  "risk_score": 88, "fraud_flag": False, "stock": 5, "quantity": 1},
    {"order_id": "ORD-103", "payment_confirmed": True,  "risk_score": 15, "fraud_flag": True,  "stock": 5, "quantity": 1},
    {"order_id": "ORD-104", "payment_confirmed": True,  "risk_score": 20, "fraud_flag": False, "stock": 0, "quantity": 2},
    {"order_id": "ORD-105", "payment_confirmed": False, "risk_score": 10, "fraud_flag": False, "stock": 5, "quantity": 1},
]
assert len({o['order_id'] for o in ORDERS}) == len(ORDERS)
assert all(o['quantity'] > 0 and 0 <= o['risk_score'] <= 100 for o in ORDERS)
print("PASS: 5 fictional orders and business rules ready")
print("Risk hold threshold:", RULES["risk_hold_threshold"])

## Cell 3 — Directed Graph बनाएँ
Graph का नक्शा: `START → PAYMENT_CHECK → RISK_CHECK → FRAUD_CHECK → INVENTORY_CHECK → SHIPPING → END`. कुछ Nodes से `HOLD` या `REFUND` Branch भी निकलती है।

In [ ]:
# Cell 3 — Directed adjacency graph (pure Python: no extra library)
GRAPH = {
    "START": ["PAYMENT_CHECK"],
    "PAYMENT_CHECK": ["RISK_CHECK", "HOLD"],
    "RISK_CHECK": ["FRAUD_CHECK", "HOLD"],
    "FRAUD_CHECK": ["INVENTORY_CHECK", "HOLD"],
    "INVENTORY_CHECK": ["SHIPPING", "REFUND"],
    "SHIPPING": ["END"],
    "REFUND": ["END"],
    "HOLD": ["END"],
    "END": [],
}
assert all(next_node in GRAPH for options in GRAPH.values() for next_node in options)
assert len(GRAPH) == 9
print("PASS: Directed Graph created (9 nodes)")
for node, destinations in GRAPH.items():
    print(f"{node:16} -> {', '.join(destinations) if destinations else '(finish)'}")

## Cell 4 — Graph Router का Python Function
अगले Node का चुनाव **Python Rules** करेंगे। LLM कभी Payment या Refund execute नहीं करेगा।

In [ ]:
# Cell 4 — Deterministic graph routing

def next_node(node, order):
    if node == "START":
        return "PAYMENT_CHECK"
    if node == "PAYMENT_CHECK":
        return "RISK_CHECK" if order["payment_confirmed"] else "HOLD"
    if node == "RISK_CHECK":
        return "HOLD" if order["risk_score"] >= RULES["risk_hold_threshold"] else "FRAUD_CHECK"
    if node == "FRAUD_CHECK":
        return "HOLD" if order["fraud_flag"] else "INVENTORY_CHECK"
    if node == "INVENTORY_CHECK":
        return "SHIPPING" if order["stock"] >= order["quantity"] else "REFUND"
    if node in ("SHIPPING", "REFUND", "HOLD"):
        return "END"
    raise ValueError(f"Unknown node: {node}")


def run_graph(order):
    node = "START"
    path = [node]
    for _ in range(len(GRAPH) + 1):
        if node == "END":
            break
        following = next_node(node, order)
        if following not in GRAPH[node]:
            raise AssertionError(f"Invalid graph edge: {node} -> {following}")
        node = following
        path.append(node)
    else:
        raise RuntimeError("Graph did not reach END")

    outcome = next(n for n in ("SHIPPING", "REFUND", "HOLD") if n in path)
    return {"order_id": order["order_id"], "outcome": outcome, "path": path}

print("PASS: Graph router configured (no AI calls yet)")

## Cell 5 — पाँच Orders को Graph से Route करें (Offline Test)
यह Cell **बिना Ollama inference** के भी चलता है। Expected: 1 Shipping, 1 Refund और 3 Hold।

In [ ]:
# Cell 5 — Verify all graph branches deterministically
traces = [run_graph(order) for order in ORDERS]
expected = {
    "ORD-101": "SHIPPING",  # clean order
    "ORD-102": "HOLD",      # high risk
    "ORD-103": "HOLD",      # fraud flag
    "ORD-104": "REFUND",    # insufficient stock (simulation only)
    "ORD-105": "HOLD",      # payment pending
}
for result in traces:
    assert result["outcome"] == expected[result["order_id"]]
    assert result["path"][0] == "START" and result["path"][-1] == "END"
    print(result["order_id"], "|", result["outcome"], "|", " -> ".join(result["path"]))
print("PASS: All 5 orders routed via correct graph branches")

## Cell 6 — Ollama से AI Agents की व्याख्या लेने का Function
Risk, Fraud और Inventory—तीनों Stage पर **एक ही Local Qwen3 मॉडल, अलग विशेषज्ञ Prompt** इस्तेमाल होगा।

In [ ]:
# Cell 6 — Local Ollama API client and specialist prompts

def ollama_chat(system_prompt, user_prompt, max_tokens=145):
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "stream": False,
        "think": False,
        "keep_alive": "10m",
        "options": {"temperature": 0.2, "num_predict": max_tokens, "num_ctx": 2048},
    }
    req = Request(
        f"{OLLAMA_URL}/api/chat",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urlopen(req, timeout=240) as response:
            data = json.load(response)
    except HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")[:400]
        raise RuntimeError(f"Ollama HTTP {exc.code}: {detail}") from exc
    except (URLError, TimeoutError, OSError) as exc:
        raise RuntimeError("Ollama Model से संपर्क नहीं हुआ। Ollama App चालू रखें।") from exc
    answer = data.get("message", {}).get("content", "").strip()
    if not answer:
        raise RuntimeError("AI ने खाली जवाब दिया। Ollama की स्थिति जाँचें।")
    return answer

AGENT_ROLES = {
    "RISK_CHECK": "You are a risk review agent. Explain supplied order risk facts in 1-2 simple Hindi sentences. Do not change the risk score, invent facts, or make the routing decision.",
    "FRAUD_CHECK": "You are a fraud review agent. Explain the given fraud flag in 1-2 simple Hindi sentences. Do not invent fraud evidence or make routing decisions.",
    "INVENTORY_CHECK": "You are an inventory agent. Explain the given stock and quantity in 1-2 simple Hindi sentences. Do not claim to ship goods or change stock.",
}
print("PASS: Three local AI specialist roles configured")

## Cell 7 — Real Qwen3 Agents को Graph के Nodes पर चलाएँ
अब `ORD-101` के लिए Risk, Fraud और Inventory Agents **3 वास्तविक Local AI Calls** करेंगे। निर्णय Python के Graph Router से पहले ही तय हैं। Small model पर कुछ समय लग सकता है।

In [ ]:
# Cell 7 — Attach real local AI explanations to graph nodes
selected_order = ORDERS[0]  # clean fictional order: all three checks visited
selected_trace = run_graph(selected_order)
agent_notes = []
for node in selected_trace["path"]:
    if node in AGENT_ROLES:
        facts = json.dumps(selected_order, ensure_ascii=False)
        question = (
            f"Fictional order facts: {facts}\n"
            f"Current graph step: {node}. The Python router selected: {selected_trace['outcome']}. "
            "Explain the supplied facts briefly in Hindi. This is a simulation only."
        )
        answer = ollama_chat(AGENT_ROLES[node], question)
        agent_notes.append({"node": node, "agent_answer": answer})
        print(f"\n--- {node} Agent ---\n{answer}\n")

assert len(agent_notes) == 3
assert run_graph(selected_order) == selected_trace  # AI did not change any route
print("PASS: 3 local Qwen3 specialists explained Graph nodes; routing unchanged")

## Cell 8 — Final Report और Graph Traces Save करें
File Browser में **2 Files** आएँगी: `.md` Report और `.json` Graph Traces।

In [ ]:
# Cell 8 — Save local files in the current Jupyter working directory
md_path = Path("section4_graph_pattern_report.md")
json_path = Path("section4_graph_pattern_traces.json")

lines = [
    "# Section 4 — Practical 3: Graph Pattern (Ollama + Qwen3)",
    "",
    "Educational simulation with fictional orders. No real payment, shipping or refund.",
    "",
    "## Routed orders",
]
for item in traces:
    lines.extend([
        f"- **{item['order_id']}**: {item['outcome']}",
        f"  - Route: {' → '.join(item['path'])}",
    ])
lines.extend(["", "## Local AI specialist explanations — ORD-101", ""])
for note in agent_notes:
    lines.extend([f"### {note['node']}", note["agent_answer"], ""])

md_path.write_text("\n".join(lines), encoding="utf-8")
json_path.write_text(json.dumps({"rules": RULES, "orders": ORDERS, "traces": traces, "local_ai_notes": agent_notes}, ensure_ascii=False, indent=2), encoding="utf-8")
assert md_path.is_file() and json_path.is_file()
print("PASS: Section 4 Practical 3 — Graph Pattern completed")
print("Saved:", md_path.resolve())
print("Saved:", json_path.resolve())

---
## यदि Error आए
- **Cell 1 — Connection refused:** Windows में Ollama App खोलें, फिर Cell 1 दोबारा चलाएँ।
- **Cell 1 — Qwen Model missing:** PowerShell में `ollama pull qwen3:1.7b` चलाएँ।
- **Cell 7 — Timeout / Ollama stopped:** Ollama खुला रखें; Cell 7 फिर चलाएँ।
- **Cell 8 — NameError:** Cell 1 से 7 क्रम से चलाएँ।
- **`In [*]` दिखाई दे:** आपका Local Qwen3 जवाब बना रहा है।

**क्या सीखा?** `Graph Pattern` में पहले से तय Nodes और Conditional Edges होते हैं। `Supervisor–Worker` delegation से अलग, Workflow के अगले Step का फैसला Graph Router कर रहा है। यहाँ LLM specialists प्रत्येक Node की व्याख्या देते हैं, पर वह असली भुगतान, रिफंड या डिलीवरी नहीं करते।